# 1. Mesh preprocessing

First notebook of the pipeline (`preprocess_mesh` → `la_construction` → `la_basis` → `la_mapping`). It turns a raw left-atrium (LA) volumetric mesh with fiber orientations and legacy anatomical tags into a clean triangular surface mesh that the later notebooks can consume.

Steps:
1. Read the raw `.vtu` and extract the surface as triangular `PolyData`.
2. Drop stale `alpha`/`beta` point data and re-label the anatomical tags to the pipeline's convention.
3. Run `mesh_utils.preprocess_mesh` (optional decimation/smoothing, region extraction, unit scaling).
4. Save the result and inspect the fiber field.

In [ ]:
from pathlib import Path

import numpy as np
import pyvista as pv

from uacpy.common import mesh_utils
from uacpy.output import visualization

## Parameters

- `decimation_factor` / `num_smoothing_iterations`: optional mesh reduction and smoothing (`1` and `0` leave the mesh as is).
- `extraction_threshold`: threshold passed to `preprocess_mesh` for extracting the region of interest.
- `dimension_scaling_factor`: multiplies the patient-space coordinates (here `1e-3`) to rescale the mesh to the units used downstream.
- `legacy_tags` → `new_tags`: maps the tag values in the raw data to the ones the pipeline expects (`MV`=mitral valve, `LAA`=appendage, `LIPV`/`LSPV`/`RIPV`/`RSPV`=left/right inferior/superior pulmonary veins).

In [ ]:
patient_id = "02"
input_mesh_file = Path(f"../data/patient_{patient_id}/mesh_endo_raw.vtu")
output_mesh_file = Path("test_mesh.vtu")
decimation_factor = 1
num_smoothing_iterations = 0
extraction_threshold = 0.75
dimension_scaling_factor = 1e-3
legacy_tags = {"MV": 11, "LAA": 13, "LIPV": 21, "LSPV": 23, "RIPV": 25, "RSPV": 27}
new_tags = {"MV": 0, "LAA": 1, "LIPV": 2, "LSPV": 3, "RIPV": 4, "RSPV": 5}

## Convert, re-tag and preprocess

The volumetric mesh is converted to a triangular surface, and the `alpha`/`beta` arrays left over from an earlier parameterization are removed so they cannot be confused with the coordinates computed later. The legacy tag values on each cell are then replaced by the new ones, the mesh is preprocessed, and the result is written to `output_mesh_file`.

In [ ]:
input_mesh = pv.read(input_mesh_file)
triangular_mesh = mesh_utils.convert_unstructured_to_polydata_mesh(input_mesh)
del triangular_mesh.point_data["alpha"]
del triangular_mesh.point_data["beta"]
for key, legacy_value in legacy_tags.items():
    anatomical_region = np.where(triangular_mesh.cell_data["anatomical_tags"] == legacy_value)[0]
    triangular_mesh.cell_data["anatomical_tags"][anatomical_region] = new_tags[key]

triangular_mesh = mesh_utils.preprocess_mesh(
    triangular_mesh,
    tag_values=new_tags.values(),
    decimation_factor=decimation_factor,
    num_smoothing_iters=num_smoothing_iterations,
    extraction_threshold=extraction_threshold,
    dimension_scaling_factor=dimension_scaling_factor,
)
output_mesh_file.parent.mkdir(parents=True, exist_ok=True)
pv.save_meshio(output_mesh_file, triangular_mesh)

## Compare the fibers before and after preprocessing

The first plot shows the raw input mesh with its fibers, the second the preprocessed mesh with the fibers carried over onto it (unit tangent vectors per cell). Arrow length is scaled by `dimension_scaling_factor` in the second plot, since the preprocessed mesh has been rescaled. The two fields should look alike.

In [ ]:
visualization.plot_mesh_with_vector_field(
    input_mesh,
    vector_field=input_mesh.cell_data["fibers"],
    scaling_factor=500,
)

In [ ]:
visualization.plot_mesh_with_vector_field(
    triangular_mesh,
    vector_field=triangular_mesh.cell_data["fibers"],
    scaling_factor=500 * dimension_scaling_factor,
)